In [0]:
customers = spark.read.csv("/Workspace/Users/iluqmanshaik255@gmail.com/customers_phase4.csv" , inferSchema = True , header = True)

sales = spark.read.csv("/Workspace/Users/iluqmanshaik255@gmail.com/sales_phase4.csv" , inferSchema = True , header = True)

In [0]:
customers.printSchema()
customers.show()

root
 |-- customer_id: double (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- phone: long (nullable = true)
 |-- email: string (nullable = true)
 |-- join_date: date (nullable = true)

+-----------+-------------+---------+-----------+----------+------------------+----------+
|customer_id|customer_name|     city|      state|     phone|             email| join_date|
+-----------+-------------+---------+-----------+----------+------------------+----------+
|     1001.0|      Vihaan1|     Pune|Maharashtra|9813356886| vihaan1@gmail.com|2025-01-14|
|     1002.0|       Kiran2|  Chennai| Tamil Nadu|9839958838|  kiran2@gmail.com|2024-03-12|
|     1003.0|      Vihaan3|     Pune|Maharashtra|9883197857| vihaan3@gmail.com|2024-02-14|
|     1004.0|       Meena4|    Delhi|      Delhi|9814265799|  meena4@gmail.com|2024-01-16|
|     1005.0|       Rahul5|Hyderabad|  Telangana|9841227216|  rahul5@gmail.com|2024-0

In [0]:
sales.printSchema()
sales.show()

root
 |-- sale_id: integer (nullable = true)
 |-- customer_id: double (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- total_amount: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- payment_method: string (nullable = true)

+-------+-----------+------------+-----------+--------+----------+------------+----------+--------------+
|sale_id|customer_id|product_name|   category|quantity|unit_price|total_amount| sale_date|payment_method|
+-------+-----------+------------+-----------+--------+----------+------------+----------+--------------+
|      1|     1002.0|     Monitor|Electronics|       1|      4900|        4900|2025-03-09|           UPI|
|      2|     1014.0|    Keyboard|Accessories|       3|      9732|       29196|2025-06-04|          Card|
|      3|     1094.0|      Tablet|Electronics|       2|      9150|       18300|2025

In [0]:
from pyspark.sql.functions import col
for column in customers.columns:
    print(column, customers.filter(col(column).isNull()).count())

for column in sales.columns:
    print(column, sales.filter(col(column).isNull()).count())

customer_id 2
customer_name 0
city 0
state 0
phone 0
email 0
join_date 0
sale_id 0
customer_id 2
product_name 0
category 0
quantity 0
unit_price 0
total_amount 0
sale_date 0
payment_method 0


In [0]:
customers = customers.dropna(subset=["customer_id"])
sales = sales.dropna(subset=["customer_id"])

In [0]:
customers = customers.dropDuplicates()
sales = sales.dropDuplicates()


In [0]:
sales = sales.filter(
    (col("quantity") > 0) &
    (col("unit_price") > 0) &
    (col("total_amount") > 0)
)

In [0]:
customers.createOrReplaceTempView("customers")
sales.createOrReplaceTempView("sales")

In [0]:
print("Customers:", customers.count())
print("Sales:", sales.count())

Customers: 98
Sales: 496


In [0]:
%sql
SELECT
sale_date,
SUM(total_amount) AS total_sales
FROM sales
GROUP BY sale_date
ORDER BY sale_date;

sale_date,total_sales
2025-01-01,35352
2025-01-02,68241
2025-01-03,80863
2025-01-04,75450
2025-01-05,76574
2025-01-06,1226
2025-01-07,41810
2025-01-08,73472
2025-01-09,36110
2025-01-10,23051


In [0]:
from pyspark.sql.functions import sum

daily_sales = sales.groupBy("sale_date") \
    .agg(sum("total_amount").alias("total_sales")) \
    .orderBy("sale_date")

daily_sales.show()

+----------+-----------+
| sale_date|total_sales|
+----------+-----------+
|2025-01-01|      35352|
|2025-01-02|      68241|
|2025-01-03|      80863|
|2025-01-04|      75450|
|2025-01-05|      76574|
|2025-01-06|       1226|
|2025-01-07|      41810|
|2025-01-08|      73472|
|2025-01-09|      36110|
|2025-01-10|      23051|
|2025-01-11|      52489|
|2025-01-12|      89222|
|2025-01-13|      42586|
|2025-01-14|      76934|
|2025-01-16|     125287|
|2025-01-17|     151505|
|2025-01-18|     141500|
|2025-01-19|     148863|
|2025-01-20|     245318|
|2025-01-21|      26882|
+----------+-----------+
only showing top 20 rows


In [0]:
%sql
SELECT
c.city,
SUM(s.total_amount) AS total_revenue
FROM customers c
INNER JOIN sales s
ON c.customer_id=s.customer_id
GROUP BY c.city
ORDER BY total_revenue DESC;

city,total_revenue
Bengaluru,3556017
Mumbai,2920484
Delhi,2849326
Pune,2390383
Hyderabad,2298797
Chennai,1443616


In [0]:
from pyspark.sql.functions import sum

city_revenue = customers.join(
    sales,
    on="customer_id",
    how="inner"
).groupBy(
    "city"
).agg(
    sum("total_amount").alias("total_revenue")
).orderBy(
    "total_revenue",
    ascending=False
)

city_revenue.show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bengaluru|      3556017|
|   Mumbai|      2920484|
|    Delhi|      2849326|
|     Pune|      2390383|
|Hyderabad|      2298797|
|  Chennai|      1443616|
+---------+-------------+



In [0]:
%sql
SELECT
c.customer_name,
SUM(s.total_amount) AS total_spend
FROM customers c
INNER JOIN sales s
ON c.customer_id=s.customer_id
GROUP BY c.customer_name
ORDER BY total_spend DESC
LIMIT 5;

customer_name,total_spend
Vivaan61,425340
Nikhil99,392494
Priya44,353963
Meena54,342007
Aditya51,338887


In [0]:
from pyspark.sql.functions import sum

top5 = customers.join(
    sales,
    on="customer_id",
    how="inner"
).groupBy(
    "customer_name"
).agg(
    sum("total_amount").alias("total_spend")
).orderBy(
    "total_spend",
    ascending=False
).limit(5)

top5.show()

+-------------+-----------+
|customer_name|total_spend|
+-------------+-----------+
|     Vivaan61|     425340|
|     Nikhil99|     392494|
|      Priya44|     353963|
|      Meena54|     342007|
|     Aditya51|     338887|
+-------------+-----------+



In [0]:
%sql
SELECT
customer_id,
COUNT(sale_id) AS order_count
FROM sales
GROUP BY customer_id
HAVING COUNT(sale_id) > 1
ORDER BY order_count DESC;

customer_id,order_count
1006.0,13
1036.0,11
1051.0,10
1044.0,10
1086.0,10
1061.0,10
1099.0,10
1014.0,9
1011.0,9
1043.0,9


In [0]:
from pyspark.sql.functions import count

repeat_customers = sales.groupBy("customer_id") \
    .agg(count("sale_id").alias("order_count")) \
    .filter("order_count > 1") \
    .orderBy("order_count", ascending=False)

repeat_customers.show()

+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
|     1006.0|         13|
|     1036.0|         11|
|     1061.0|         10|
|     1051.0|         10|
|     1086.0|         10|
|     1044.0|         10|
|     1099.0|         10|
|     1043.0|          9|
|     1070.0|          9|
|     1014.0|          9|
|     1011.0|          9|
|     1012.0|          9|
|     1054.0|          9|
|     1062.0|          8|
|     1081.0|          8|
|     1034.0|          8|
|     1058.0|          8|
|     1030.0|          8|
|     1066.0|          8|
|     1067.0|          7|
+-----------+-----------+
only showing top 20 rows


In [0]:
%sql
SELECT
c.customer_name,
SUM(s.total_amount) AS total_spend,
CASE
    WHEN SUM(s.total_amount) > 10000 THEN 'Gold'
    WHEN SUM(s.total_amount) BETWEEN 5000 AND 10000 THEN 'Silver'
    ELSE 'Bronze'
END AS segment
FROM customers c
INNER JOIN sales s
ON c.customer_id=s.customer_id
GROUP BY c.customer_name
ORDER BY total_spend DESC;

customer_name,total_spend,segment
Vivaan61,425340,Gold
Nikhil99,392494,Gold
Priya44,353963,Gold
Meena54,342007,Gold
Aditya51,338887,Gold
Anjali62,324466,Gold
Aarav6,322067,Gold
Aditya58,318739,Gold
Vihaan74,304578,Gold
Meena66,297351,Gold


In [0]:
from pyspark.sql.functions import sum, when, col

customer_segment = customers.join(
    sales,
    on="customer_id",
    how="inner"
).groupBy(
    "customer_id",
    "customer_name"
).agg(
    sum("total_amount").alias("total_spend")
).withColumn(
    "segment",
    when(col("total_spend") > 10000, "Gold")
    .when((col("total_spend") >= 5000) & (col("total_spend") <= 10000), "Silver")
    .otherwise("Bronze")
).orderBy(
    "total_spend",
    ascending=False
)

customer_segment.show()

+-----------+-------------+-----------+-------+
|customer_id|customer_name|total_spend|segment|
+-----------+-------------+-----------+-------+
|     1061.0|     Vivaan61|     425340|   Gold|
|     1099.0|     Nikhil99|     392494|   Gold|
|     1044.0|      Priya44|     353963|   Gold|
|     1054.0|      Meena54|     342007|   Gold|
|     1051.0|     Aditya51|     338887|   Gold|
|     1062.0|     Anjali62|     324466|   Gold|
|     1006.0|       Aarav6|     322067|   Gold|
|     1058.0|     Aditya58|     318739|   Gold|
|     1074.0|     Vihaan74|     304578|   Gold|
|     1066.0|      Meena66|     297351|   Gold|
|     1022.0|      Sneha22|     279806|   Gold|
|     1041.0|        Sai41|     277569|   Gold|
|     1070.0|     Anjali70|     275557|   Gold|
|     1056.0|      Rohit56|     263180|   Gold|
|     1081.0|      Arjun81|     256070|   Gold|
|     1034.0|      Divya34|     252687|   Gold|
|     1063.0|       Neha63|     250307|   Gold|
|     1043.0|      Divya43|     248873| 

In [0]:
%sql
SELECT
c.customer_name,
c.city,
SUM(s.total_amount) AS total_spend,
COUNT(s.sale_id) AS order_count,
CASE
    WHEN SUM(s.total_amount) > 10000 THEN 'Gold'
    WHEN SUM(s.total_amount) BETWEEN 5000 AND 10000 THEN 'Silver'
    ELSE 'Bronze'
END AS segment
FROM customers c
INNER JOIN sales s
ON c.customer_id=s.customer_id
GROUP BY c.customer_name,c.city
ORDER BY total_spend DESC;

customer_name,city,total_spend,order_count,segment
Vivaan61,Pune,425340,10,Gold
Nikhil99,Bengaluru,392494,10,Gold
Priya44,Pune,353963,10,Gold
Meena54,Delhi,342007,9,Gold
Aditya51,Mumbai,338887,10,Gold
Anjali62,Hyderabad,324466,8,Gold
Aarav6,Delhi,322067,13,Gold
Aditya58,Bengaluru,318739,8,Gold
Vihaan74,Delhi,304578,7,Gold
Meena66,Mumbai,297351,8,Gold


In [0]:
from pyspark.sql.functions import sum, count, when, col

final_df = customers.join(
    sales,
    on="customer_id",
    how="inner"
).groupBy(
    "customer_id",
    "customer_name",
    "city"
).agg(
    sum("total_amount").alias("total_spend"),
    count("sale_id").alias("order_count")
).withColumn(
    "segment",
    when(col("total_spend") > 10000, "Gold")
    .when((col("total_spend") >= 5000) & (col("total_spend") <= 10000), "Silver")
    .otherwise("Bronze")
).select(
    "customer_name",
    "city",
    "total_spend",
    "order_count",
    "segment"
).orderBy(
    "total_spend",
    ascending=False
)

final_df.show()

+-------------+---------+-----------+-----------+-------+
|customer_name|     city|total_spend|order_count|segment|
+-------------+---------+-----------+-----------+-------+
|     Vivaan61|     Pune|     425340|         10|   Gold|
|     Nikhil99|Bengaluru|     392494|         10|   Gold|
|      Priya44|     Pune|     353963|         10|   Gold|
|      Meena54|    Delhi|     342007|          9|   Gold|
|     Aditya51|   Mumbai|     338887|         10|   Gold|
|     Anjali62|Hyderabad|     324466|          8|   Gold|
|       Aarav6|    Delhi|     322067|         13|   Gold|
|     Aditya58|Bengaluru|     318739|          8|   Gold|
|     Vihaan74|    Delhi|     304578|          7|   Gold|
|      Meena66|   Mumbai|     297351|          8|   Gold|
|      Sneha22|Bengaluru|     279806|          7|   Gold|
|        Sai41|   Mumbai|     277569|          7|   Gold|
|     Anjali70|     Pune|     275557|          9|   Gold|
|      Rohit56|     Pune|     263180|          6|   Gold|
|      Arjun81

In [0]:
final_df.write \
    .mode("overwrite") \
    .saveAsTable("final_report")

In [0]:
%sql
SELECT * FROM final_report;

customer_name,city,total_spend,order_count,segment
Vivaan61,Pune,425340,10,Gold
Nikhil99,Bengaluru,392494,10,Gold
Priya44,Pune,353963,10,Gold
Meena54,Delhi,342007,9,Gold
Aditya51,Mumbai,338887,10,Gold
Anjali62,Hyderabad,324466,8,Gold
Aarav6,Delhi,322067,13,Gold
Aditya58,Bengaluru,318739,8,Gold
Vihaan74,Delhi,304578,7,Gold
Meena66,Mumbai,297351,8,Gold
